# Collecting Data with APIs
## HPI · Python practical · 6 November 2026

**By the end:** explain an API request, collect video metadata, turn nested JSON into a table, and describe what your dataset covers.

We reuse the core of Juan S. Gómez-Cruces’s supplied YouTube notebook: **video search → video IDs → metadata → pandas**. Channel discovery is an optional extension. Audio download and Whisper transcription are outside this session.

**Class plan:** 25 minutes of concepts; 10 minutes to get ready; 40 minutes of collection and exploration; 15 minutes of voluntary sharing.

### Get ready
1. Open [github.com/manueltonneau/hpi-api](https://github.com/manueltonneau/hpi-api) and click **Open in Colab**, or download this notebook for Jupyter. In Colab, use **File → Save a copy in Drive** to keep your work.
2. For live collection, create your own YouTube Data API v3 key: [step-by-step setup](https://github.com/manueltonneau/hpi-api/blob/main/SETUP.md). Google account → Cloud project → enable YouTube Data API v3 → Credentials → API key. Public data uses a standard API key, not OAuth or a service-account key.
3. In the next cell, set `OFFLINE=False` for live collection. The private key input is hidden. With `OFFLINE=True`, you can do the guided example using the saved records while resolving setup.
4. After the guided example, choose a question and a request template. You can collect videos, channel statistics or a small set of top-level comments.

### Run the notebook
- Python 3.10+ and `pandas`; `requests` for live requests. Run `%pip install pandas requests` in a separate cell if needed.
- Run cells in order with Shift+Enter. Change only the marked experiment settings first.
- **Offline is the default.** Ten real saved records are embedded. The helper reconstructs small API-shaped teaching responses from them. These are **not original raw responses or replayed searches**. No key or network is needed.
- **Live:** set `OFFLINE = False`; use your own YouTube Data API v3 key. If no environment variable is set, a private input box will appear. Never put the key in a code cell or print a request URL.
- The original example studies a German political channel. Keep it for a reproducible exercise; live groups can choose another verified channel ID. Nothing here is a representative sample of YouTube.

**Your project:** after the guided example, choose the question, request parameters and fields you want to summarise. Sharing a result with the class is optional; you may work and share in pairs.


In [ ]:
import os
import json
import time
from getpass import getpass
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

OFFLINE = True
CHANNEL_ID = "UC_dZp8bZipnjntBGLVHm6rw"  # Juan's example channel
PAGE_SIZE = 5
BASE_ORDER = "viewCount"
OUTPUT_DIR = Path("workshop_outputs")
API_KEY = os.environ.get("YOUTUBE_API_KEY", "")
if not OFFLINE and not API_KEY:
    API_KEY = getpass("YouTube API key (input hidden): ").strip()
if not OFFLINE and not API_KEY:
    raise ValueError("A key is required for live YouTube requests. Use OFFLINE=True as a fallback.")
if not 1 <= PAGE_SIZE <= 50:
    raise ValueError("Use PAGE_SIZE from 1 to 50.")
print("Mode:", "OFFLINE teaching fixtures" if OFFLINE else "LIVE YouTube API")


### Provided setup: saved records and request helper
Run the next two cells once. Their source is collapsed where your notebook viewer supports it; you do not need to write this code.

Offline fixtures are reconstructed from the displayed outputs in Juan’s notebook (zero-based cells 12 and 14). The exact original observation time is unknown. Publication time is not collection time. Live responses are kept separately with observation timestamps.

The transport underneath the helper is standard Python:

```python
response = requests.get(url, params=parameters, timeout=(5, 25))
payload = response.json()
```

The helper also checks the HTTP status and expected fields before returning that payload. It limits retries and omits credentials from logs. It raises an error for failed requests rather than pretending they returned no records.


In [ ]:
FIXTURE = {'provenance': {'description': 'Normalized records reconstructed from displayed saved outputs, not raw API responses. Original exact observation time unknown. Video publication dates are not collection timestamps.', 'source_file': 'youtube_germany_annotated2.ipynb', 'source_cells_zero_based': [12, 14], 'records': 10, 'scope': 'Ten videos printed by the supplied notebook. Coverage and original search completeness have not been verified.'}, 'records': [{'video_id': '8ioWqF4k2fo', 'title': 'Brandmauer weg oder Neuwahlen! PK mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:56Z', 'duration': 'PT23M7S', 'views': 308065, 'likes': 25574, 'comments': 2288, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'eYh7yfnluqs', 'title': 'Interview mit Beatrix von Storch zum Ergebnis des Koalitionsgipfels! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-13T15:01:09Z', 'duration': 'PT17M35S', 'views': 249565, 'likes': 20505, 'comments': 1563, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Uf9nmuTDesk', 'title': 'Regierungsbefragung mit Lars Klingbeil und Verena Hubertz! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T14:50:16Z', 'duration': 'PT17M9S', 'views': 149674, 'likes': 7771, 'comments': 1822, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Hz6SLP23Nv0', 'title': 'LIVE: Pressekonferenz der AfD-Fraktion - Diese Woche im Bundestag', 'published_at': '2026-05-19T08:26:19Z', 'duration': 'PT1H9M57S', 'views': 147981, 'likes': 11521, 'comments': 1183, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'OU-pD4Tm90w', 'title': 'BUNDESTAG LIVE - 80. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-21T21:23:45Z', 'duration': 'PT11H55M1S', 'views': 117253, 'likes': 2224, 'comments': 5, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'HbPnlbTchM8', 'title': 'Dr. Anna Rathert zum Untersuchungsausschuss Nord-Stream! - AfD-Fraktion im Deutschen Bundestag', 'published_at': '2026-05-21T10:47:14Z', 'duration': 'PT4M32S', 'views': 112037, 'likes': 7779, 'comments': 805, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '5m7dIsDftmQ', 'title': 'Bernd Baumann knöpft sich die Umfrageverlierer Söder, Maier und Banaszak vor! - AfD-Fraktion', 'published_at': '2026-05-19T09:17:11Z', 'duration': 'PT2M22S', 'views': 103756, 'likes': 7568, 'comments': 299, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'IehYwnYBwZs', 'title': 'BUNDESTAG LIVE - 79. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T18:51:36Z', 'duration': 'PT6H41M30S', 'views': 98033, 'likes': 2274, 'comments': 25, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '40qjQvDzUII', 'title': 'Nordstream: Wir fordern einen Untersuchungsausschuss! Markus Frohnmaier im Interview - AfD-Fraktion', 'published_at': '2026-05-20T16:00:05Z', 'duration': 'PT12M53S', 'views': 81091, 'likes': 7003, 'comments': 695, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'iwycKryn3aw', 'title': 'LIVE: Presseerklärung mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:22Z', 'duration': 'PT23M32S', 'views': 72894, 'likes': 6037, 'comments': 612, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}]}

OFFLINE_PAGE = "offline-page:"

def offline_payload(endpoint, parameters):
    if CHANNEL_ID != "UC_dZp8bZipnjntBGLVHm6rw":
        raise ValueError("Offline data contain only Juan's example channel. Use live mode for another channel.")
    records = FIXTURE["records"]
    if endpoint == "search":
        if parameters.get("channelId") != CHANNEL_ID:
            raise ValueError("The offline cache cannot answer a request for another channel.")
        if parameters.get("type") != "video" or parameters.get("q"):
            raise ValueError("Offline fixtures support video searches without keywords only.")
        order = parameters.get("order", "viewCount")
        if order not in {"date", "viewCount"}:
            raise ValueError("Offline exercise supports date or viewCount ordering.")
        field = "published_at" if order == "date" else "views"
        ranked = sorted(records, key=lambda r: r[field], reverse=True)
        token = parameters.get("pageToken", OFFLINE_PAGE + "0")
        if not token.startswith(OFFLINE_PAGE):
            raise ValueError("Unexpected offline page token.")
        start, size = int(token[len(OFFLINE_PAGE):]), int(parameters["maxResults"])
        selected = ranked[start:start + size]
        payload = {"items": [{"id": {"videoId": r["video_id"]},
                              "snippet": {"title": r["title"]}} for r in selected]}
        if start + size < len(ranked):
            payload["nextPageToken"] = OFFLINE_PAGE + str(start + size)
        return payload
    if endpoint == "videos":
        wanted = parameters.get("id", "").split(",")
        by_id = {r["video_id"]: r for r in records}
        return {"items": [{"id": v,
            "snippet": {"title": by_id[v]["title"], "publishedAt": by_id[v]["published_at"],
                        "channelId": by_id[v]["channel_id"]},
            "statistics": {api: str(by_id[v][local]) for api, local in
                           [("viewCount", "views"), ("likeCount", "likes"), ("commentCount", "comments")]
                           if by_id[v][local] is not None},
            "contentDetails": {"duration": by_id[v]["duration"]}}
            for v in wanted if v in by_id]}
    raise ValueError("This endpoint is not available in offline fixtures.")

print("Loaded", len(FIXTURE["records"]), "saved records; no network requests.")


In [ ]:
RAW_RESPONSES = []
REQUEST_LOG = []

def api_get(endpoint, parameters):
    if OFFLINE:
        return offline_payload(endpoint, parameters)
    if endpoint not in {"search", "videos", "channels", "playlistItems", "commentThreads"}:
        raise ValueError("This workshop exposes only read endpoints.")
    import requests
    safe_parameters = {k: v for k, v in parameters.items() if k != "key"}
    parameters = dict(safe_parameters, key=API_KEY)
    for attempt in range(3):
        requested_at = datetime.now(timezone.utc).isoformat()
        try:
            response = requests.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params=parameters, timeout=(5, 25),
            )
        except requests.RequestException as exc:
            raise RuntimeError(f"Network request failed ({type(exc).__name__}); credentials omitted.") from None
        REQUEST_LOG.append({"endpoint": endpoint, "parameters": safe_parameters,
                            "status": response.status_code,
                            "requested_at": requested_at})
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 2:
            retry_after = response.headers.get("Retry-After", "")
            delay = min(float(retry_after), 30) if retry_after.isdigit() else 2 ** attempt
            time.sleep(delay)
            continue
        try:
            payload = response.json()
        except ValueError:
            raise RuntimeError(f"HTTP {response.status_code}: response was not valid JSON.") from None
        if not isinstance(payload, dict):
            raise RuntimeError("Unexpected API response: expected a JSON object.")
        if response.status_code != 200 or "error" in payload:
            reasons = [str(e.get("reason", "unknown"))
                       for e in payload.get("error", {}).get("errors", [])]
            raise RuntimeError(f"API request stopped: HTTP {response.status_code}, reasons={reasons}")
        if not isinstance(payload.get("items"), list):
            raise RuntimeError("Unexpected API response: expected an items list.")
        RAW_RESPONSES.append({"endpoint": endpoint, "parameters": safe_parameters,
                              "received_at": datetime.now(timezone.utc).isoformat(),
                              "payload": payload})
        return payload
    raise RuntimeError("Request did not succeed within the retry budget.")


### 1. Search for video IDs (guided, about 5 minutes)
An **endpoint** is an API address for a particular operation. `GET /search` retrieves matches. Parameters specify the type, channel, ordering and page size; the helper adds your key in live mode.

Before running, identify which parameter you would change to request fewer videos. Read the [search documentation](https://developers.google.com/youtube/data/reference/rest/v3/search/list) if you need a hint.


In [ ]:
search_params = {
    "part": "snippet",
    "channelId": CHANNEL_ID,
    "type": "video",
    "order": BASE_ORDER,
    "maxResults": PAGE_SIZE,
}
search_data = api_get("search", search_params)
print("Top-level fields:", list(search_data))
print("Matches on this page:", len(search_data["items"]))
if search_data["items"]:
    print(json.dumps(search_data["items"][0], indent=2, ensure_ascii=False))
print("Another page available:", "nextPageToken" in search_data)


**Try:** locate the first video ID in the nested result. `items` is a list; each item is a dictionary. Search gives identifiers and snippets, not view counts.


In [ ]:
video_ids = list(dict.fromkeys(item["id"]["videoId"] for item in search_data["items"]))
print(video_ids)


### 2. Retrieve metadata for those IDs (guided, about 5 minutes)
The `videos` endpoint returns details for specified video IDs. Requesting several IDs together avoids one request per video.

**Predict:** where in this response will the view count be? Is it a number or a string?


In [ ]:
stats_params = {
    "part": "snippet,statistics,contentDetails",
    "id": ",".join(video_ids),
}
stats_data = api_get("videos", stats_params) if video_ids else {"items": []}
if stats_data["items"]:
    print(json.dumps(stats_data["items"][0], indent=2, ensure_ascii=False))
returned_ids = {item["id"] for item in stats_data["items"]}
print("IDs without returned metadata:", sorted(set(video_ids) - returned_ids))


### 3. Build a table (guided, about 5 minutes)
JSON means **JavaScript Object Notation**. Here `response.json()` becomes Python dictionaries and lists. Structure lets us address a value directly, e.g. `item["statistics"]["viewCount"]`; it does not remove all cleaning.

The following closely follows Juan’s dataframe cell. We keep the video ID, convert count strings, parse dates, and preserve unavailable counts as missing rather than zero. `PT23M7S` is an ISO 8601 duration, not a clock time.


In [ ]:
def make_table(payload):
    rows = [{
        "video_id": item["id"],
        "title": item.get("snippet", {}).get("title"),
        "published_at": item.get("snippet", {}).get("publishedAt"),
        "channel_id": item.get("snippet", {}).get("channelId"),
        "duration": item.get("contentDetails", {}).get("duration"),
        "views": item.get("statistics", {}).get("viewCount"),
        "likes": item.get("statistics", {}).get("likeCount"),
        "comments": item.get("statistics", {}).get("commentCount"),
    } for item in payload["items"]]
    columns = ["video_id", "title", "published_at", "channel_id", "duration", "views", "likes", "comments"]
    frame = pd.DataFrame(rows, columns=columns)
    for field in ["views", "likes", "comments"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    frame["published_at"] = pd.to_datetime(frame["published_at"], utc=True)
    return frame

baseline_df = make_table(stats_data)
print(baseline_df[["video_id", "title", "views", "comments"]].to_string(index=False))


### 4. Your collection project (about 20–25 minutes)
Work alone or in pairs. Choose a small question, then a request that could help answer it. You do not have to reproduce the instructor’s channel or the ordering comparison.

| Starting point | Unit | Choices you can make |
| --- | --- | --- |
| `channel_videos` | Video | Channel, keyword restriction, publication window, ordering, page size |
| `topic_videos` | Video | Search phrase, publication window, ordering, optional channel restriction |
| `channel_statistics` (live only) | Channel | One or more verified channel IDs; public video count, subscribers, total views |
| `video_comments` (live only) | Top-level comment | Video ID, recent/relevance order, page size; text and counts |

Possible questions: What do the returned videos about a topic look like? How do views vary within a channel’s selected videos? How do a few chosen channels compare in their reported statistics? These are bounded collections, not complete platform samples.

**Before collecting, write:**
- Our question is …
- Our unit is a video / channel / top-level comment.
- We need the following fields …
- We will request them using …

Keep the collection small: **at most 20 records per search page and two pages**. Modify one setting at a time. Use [search docs](https://developers.google.com/youtube/data/reference/rest/v3/search/list), [video docs](https://developers.google.com/youtube/data/reference/rest/v3/videos/list), [channel docs](https://developers.google.com/youtube/data/reference/rest/v3/channels/list), or [comment docs](https://developers.google.com/youtube/data/reference/rest/v3/commentThreads/list).

**Offline:** you can choose ordering, size, a local title filter and publication window within the ten saved videos. The filters are a local exercise, not an emulation of YouTube search. Different channels, channel-level statistics and comments require live access; those records are not in the cache.

**Channel IDs:** use the optional channel finder directly below if you need an ID. For a video ID, use the part after `v=` in a YouTube watch URL, or an ID from your returned video table.


#### Optional: find a channel ID (live only)
Set the switch below to True, search for a channel, then copy a verified ID into your chosen template. This reuses Juan’s channel-discovery step.


In [ ]:
RUN_CHANNEL_DISCOVERY = False
CHANNEL_SEARCH = "AfD-Fraktion Bundestag"  # Change to your own topic/name.
if RUN_CHANNEL_DISCOVERY and not OFFLINE:
    candidates = api_get("search", {"part": "snippet", "type": "channel", "q": CHANNEL_SEARCH, "maxResults": 5})
    for item in candidates["items"]:
        print(item["id"]["channelId"], item["snippet"]["title"])
else:
    print("Optional live channel discovery disabled.")


In [ ]:
# Choose and edit ONE of these request templates.
PROJECT_KIND = "channel_videos"  # Or "topic_videos" / "channel_statistics" / "video_comments"
PROJECT_MAX_PAGES = 1           # At most 2; applies to video search and comments

channel_video_request = {
    "part": "snippet", "type": "video", "channelId": CHANNEL_ID,
    "order": "date", "maxResults": 10,
}
topic_video_request = {
    "part": "snippet", "type": "video", "q": "climate policy",
    "order": "date", "maxResults": 10,
}
channel_statistics_request = {
    "part": "snippet,statistics", "id": CHANNEL_ID,
    # For multiple channels use a comma-separated list of verified IDs.
}
comment_request = {
    "part": "snippet", "videoId": str(baseline_df.video_id.iloc[0]) if not baseline_df.empty else "PASTE_VIDEO_ID",
    "order": "time", "maxResults": 20, "textFormat": "plainText",
}
PROJECT_PARAMS = {
    "channel_videos": channel_video_request,
    "topic_videos": topic_video_request,
    "channel_statistics": channel_statistics_request,
    "video_comments": comment_request,
}[PROJECT_KIND].copy()

# Optional video-search edits (live API; offline approximates title/date filtering):
# PROJECT_PARAMS["q"] = "interview"
# PROJECT_PARAMS["publishedAfter"] = "2026-01-01T00:00:00Z"
# PROJECT_PARAMS["publishedBefore"] = "2027-01-01T00:00:00Z"
# PROJECT_PARAMS["order"] = "viewCount"
# Metadata sections to retrieve for video IDs:
VIDEO_PARTS = "snippet,statistics,contentDetails"


**How to read these settings:** `q` is a search expression; `channelId` restricts the source; `publishedAfter` / `publishedBefore` set a publication window; `order` changes ranking; `maxResults` sets page size. A search is not automatically exhaustive. `relevanceLanguage` is a relevance preference, not a strict language filter, and `regionCode` concerns viewability, not creator nationality.

For comments, `videoId` chooses one video, `order="time"` requests recent threads and `order="relevance"` uses relevance ordering. We collect **top-level comments only**, not every reply. The default request omits author identifiers; text itself may still contain personal information. Prefer aggregate summaries when sharing, rather than identifying quotes. Some videos have comments disabled; choose another video or project if so.

The following provided collector handles the endpoint sequences and converts their responses to tables. Run it; you do not need to write it. You may inspect its source if you want to adapt it.


In [ ]:
def make_comment_table(payload, video_id):
    rows = []
    for thread in payload["items"]:
        outer = thread.get("snippet", {})
        comment = outer.get("topLevelComment", {})
        snippet = comment.get("snippet", {})
        rows.append({"comment_id": comment["id"], "video_id": outer.get("videoId", video_id),
                     "text": snippet.get("textDisplay"), "published_at": snippet.get("publishedAt"),
                     "likes": snippet.get("likeCount"), "reply_count": outer.get("totalReplyCount")})
    frame = pd.DataFrame(rows, columns=["comment_id", "video_id", "text", "published_at", "likes", "reply_count"])
    for field in ["likes", "reply_count"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    frame["published_at"] = pd.to_datetime(frame["published_at"], utc=True)
    return frame.drop_duplicates("comment_id", keep="first").reset_index(drop=True)

def make_channel_table(payload):
    rows = []
    for item in payload["items"]:
        stats = item.get("statistics", {})
        rows.append({"channel_id": item["id"],
                     "title": item.get("snippet", {}).get("title"),
                     "subscribers": None if stats.get("hiddenSubscriberCount") else stats.get("subscriberCount"),
                     "videos": stats.get("videoCount"), "total_views": stats.get("viewCount")})
    frame = pd.DataFrame(rows, columns=["channel_id", "title", "subscribers", "videos", "total_views"])
    for field in ["subscribers", "videos", "total_views"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    return frame

def collect_project(kind, parameters, max_pages=1):
    if kind not in {"channel_videos", "topic_videos", "channel_statistics", "video_comments"}:
        raise ValueError("Choose one of the four project kinds.")
    if not 1 <= max_pages <= 2:
        raise ValueError("Use one or two pages for this workshop.")
    params = {k: v for k, v in parameters.items() if k != "key"}
    if kind == "video_comments":
        if OFFLINE:
            raise ValueError("No comments are cached. Use a video project offline or enable live mode.")
        if not params.get("videoId") or params.get("part") != "snippet":
            raise ValueError("Use one videoId and part=snippet for comments.")
        if params.get("order", "time") not in {"time", "relevance"}:
            raise ValueError("Comment order must be time or relevance.")
        if not 1 <= int(params.get("maxResults", 20)) <= 20:
            raise ValueError("Use at most 20 comments per page.")
        params["textFormat"] = "plainText"
        params["fields"] = "nextPageToken,items(id,snippet(videoId,totalReplyCount,topLevelComment(id,snippet(textDisplay,publishedAt,likeCount))))"
        items, token, pages = [], None, 0
        base = {k: v for k, v in params.items() if k != "pageToken"}
        for _ in range(max_pages):
            query = base.copy()
            if token:
                query["pageToken"] = token
            try:
                data = api_get("commentThreads", query)
            except RuntimeError as exc:
                if "commentsDisabled" in str(exc):
                    raise RuntimeError("Comments are disabled for this video. Choose another video or another project.") from None
                raise
            items.extend(data["items"])
            pages += 1
            token = data.get("nextPageToken")
            if not token:
                break
        frame = make_comment_table({"items": items}, params["videoId"])
        return frame, {"unit": "top_level_comment", "parameters": params, "pages_read": pages,
                       "more_pages_available": bool(token), "records": len(frame),
                       "scope": "Top-level comments returned; reply text is not collected.", "coverage_verified": False}
    if kind == "channel_statistics":
        if OFFLINE:
            raise ValueError("No channel statistics are cached. Use a video project offline or enable live mode.")
        if not params.get("id") or len(params["id"].split(",")) > 20:
            raise ValueError("Use 1–20 verified channel IDs.")
        if not {"snippet", "statistics"}.issubset(set(params.get("part", "").split(","))):
            raise ValueError("Keep part=snippet,statistics for the channel table.")
        frame = make_channel_table(api_get("channels", params))
        return frame, {"unit": "channel", "parameters": params, "records": len(frame),
                       "requested_ids": params["id"].split(","),
                       "returned_ids": list(frame.channel_id), "coverage_verified": False}
    if params.get("type") != "video" or params.get("part") != "snippet":
        raise ValueError("Keep type=video and part=snippet in video search requests.")
    if kind == "channel_videos" and not params.get("channelId"):
        raise ValueError("The channel_videos template needs channelId.")
    if kind == "topic_videos" and not params.get("q"):
        raise ValueError("The topic_videos template needs a q search expression.")
    size = int(params.get("maxResults", 10))
    if not 1 <= size <= 20:
        raise ValueError("Use a page size from 1 to 20 for the project.")
    if OFFLINE:
        supported = {"part", "type", "channelId", "q", "order", "maxResults", "publishedAfter", "publishedBefore"}
        if set(params) - supported:
            raise ValueError("These extra parameters are not supported by the offline exercise.")
        if params.get("channelId", CHANNEL_ID) != "UC_dZp8bZipnjntBGLVHm6rw":
            raise ValueError("Offline data cover only the supplied example channel.")
        order = params.get("order", "date")
        if order not in {"date", "viewCount"}:
            raise ValueError("Offline mode supports date/viewCount, not relevance ranking.")
        records = list(FIXTURE["records"])
        if params.get("q"):
            records = [r for r in records if params["q"].casefold() in r["title"].casefold()]
        for field, lower in [("publishedAfter", True), ("publishedBefore", False)]:
            if params.get(field):
                boundary = pd.to_datetime(params[field], utc=True)
                records = [r for r in records if (pd.to_datetime(r["published_at"], utc=True) >= boundary
                           if lower else pd.to_datetime(r["published_at"], utc=True) < boundary)]
        ranked = sorted(records, key=lambda r: r["published_at"] if order == "date" else r["views"], reverse=True)
        ids = [r["video_id"] for r in ranked[:size * max_pages]]
        frame = make_table(offline_payload("videos", {"id": ",".join(ids)}))
        print("OFFLINE: local title/date filtering of saved records, not a replay of API search.")
        return frame, {"unit": "video", "parameters": params, "max_pages": max_pages,
                       "candidate_records": len(records), "records": len(frame),
                       "mode": "local_cache_selection", "coverage_verified": False}
    ids, token, pages = [], None, 0
    base_params = {k: v for k, v in params.items() if k != "pageToken"}
    for _ in range(max_pages):
        query = base_params.copy()
        if token:
            query["pageToken"] = token
        data = api_get("search", query)
        ids.extend(item["id"]["videoId"] for item in data["items"])
        ids = list(dict.fromkeys(ids))
        pages += 1
        token = data.get("nextPageToken")
        if not token:
            break
    items = []
    for start in range(0, len(ids), 50):
        data = api_get("videos", {"part": VIDEO_PARTS, "id": ",".join(ids[start:start + 50])})
        items.extend(data["items"])
    by_id = {item["id"]: item for item in items}
    frame = make_table({"items": [by_id[v] for v in ids if v in by_id]})
    return frame, {"unit": "video", "parameters": params, "metadata_parts": VIDEO_PARTS,
                   "pages_read": pages, "more_pages_available": bool(token), "records": len(frame),
                   "missing_metadata_ids": [v for v in ids if v not in by_id], "coverage_verified": False}


In [ ]:
project_df, project_info = collect_project(PROJECT_KIND, PROJECT_PARAMS, PROJECT_MAX_PAGES)
print(project_info)
print(project_df.to_string(index=False))


### 5. Explore and describe your data (about 5–10 minutes)
Choose a descriptive result relevant to your question. You do not need to do every suggestion:
- How many unique videos, channels or comments did you obtain? How many fields are missing?
- What are the median, mean and range of a count? Do a few large records dominate the mean?
- For a topic search, which channels account for most returned videos?
- For comments, describe text length, likes or reply counts; these refer to the returned top-level comments.
- Compare two small, deliberately selected groups, keeping their request settings comparable.

The summary cell below is a starting point. Edit or add code to answer your own question. A count of views is not a count of unique people, and published time is not the time at which a count was observed. Channel subscriber counts may be hidden or rounded; treat them as reported counts rather than exact measurements.


In [ ]:
identifier = "comment_id" if "comment_id" in project_df else "video_id" if "video_id" in project_df else "channel_id"
print("Rows:", len(project_df), "Unique IDs:", project_df[identifier].nunique())
print("Missing values by field:")
print(project_df.isna().sum())
numeric_fields = project_df.select_dtypes(include="number").columns.tolist()
if numeric_fields:
    print("Descriptive statistics:")
    print(project_df[numeric_fields].describe().T)
if "video_id" in project_df and "channel_id" in project_df and not project_df.empty:
    print("Returned videos by channel:")
    print(project_df.groupby("channel_id", dropna=False).size().sort_values(ascending=False))

if "comment_id" in project_df:
    print("Comment text length (characters):")
    print(project_df["text"].astype("string").str.len().describe())


**Optional plot:** if matplotlib is installed, make a simple chart. Choose a field available in your table. If you prefer, show your summary table instead.


In [ ]:
MAKE_PLOT = False
PLOT_FIELD = "views"  # Channels: subscribers/videos/total_views; comments: likes/reply_count.
if MAKE_PLOT:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Optional plotting needs matplotlib: %pip install matplotlib")
    else:
        if PLOT_FIELD not in numeric_fields:
            raise ValueError("Choose one of: " + ", ".join(numeric_fields))
        project_df.set_index(identifier)[PLOT_FIELD].plot.bar(rot=45, color="#B10E3C")
        plt.ylabel(PLOT_FIELD)
        plt.title("Selected records — not a platform-wide sample")
        plt.tight_layout()
        plt.show()


### 6. Save your collection and prepare a short summary
Save the table and manifest. In live mode the raw responses include timestamps; offline counts have an unknown original observation time. Keys are excluded.

Prepare four lines, or show one table/plot:
1. **Question:** what did you want to find out?
2. **Request:** which source, endpoint, filters and collection limit did you use?
3. **Result:** one descriptive finding.
4. **Limitation:** what can this collection not establish?

**Sharing is voluntary.** You can share in pairs, speak for about two minutes and use the notebook directly. No slides are required. Everyone can keep the written summary even if they prefer not to speak.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    "exercise_run_at": datetime.now(timezone.utc).isoformat(),
    "mode": "offline_teaching_fixtures" if OFFLINE else "live",
    "project_kind": PROJECT_KIND, "project": project_info,
    "guided_example": search_params, "requests": REQUEST_LOG,
    "original_observation_time": None if OFFLINE else "See received_at in api_responses.json",
    "offline_provenance": FIXTURE["provenance"] if OFFLINE else None,
    "scope": "Bounded chosen collection; no completeness or representativeness claim.",
}
project_df.to_csv(run_dir / "project_records.csv", index=False)
(run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
if not OFFLINE:
    (run_dir / "api_responses.json").write_text(json.dumps(RAW_RESPONSES, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    (run_dir / "reconstructed_guided_payloads.json").write_text(
        json.dumps({"notice": "Reconstructed teaching responses, not originals", "search": search_data,
                    "videos": stats_data}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", run_dir)


### Pagination and failed requests (reference)
For more than one search/comment page, set `PROJECT_MAX_PAGES=2` and rerun the project. The collector sends `nextPageToken` back as `pageToken`. Stop when there is no token or your budget is reached. A failed request is not an empty result: inspect the status/reason, and do not retry indefinitely.


### Sources and adaptation
Core guided workflow adapted from Juan S. Gómez-Cruces’s supplied `youtube_germany_annotated2.ipynb`: search → IDs → video metadata → dataframe. Channel discovery and channel statistics also follow its original API endpoints. The open project templates add bounded collection, private key entry, missing values and explicit provenance. Audio/transcription is outside this class.

Teaching inspiration: [Berkeley D-Lab](https://github.com/dlab-berkeley/Python-Web-APIs), [Katherine Walden](https://github.com/kwaldenphd/apis-python), and [Karsten T. Hansen / UCSD](https://bar.rady.ucsd.edu/CollectingDatawithAPI.html). Service-specific settings should be checked against the linked current official documentation.
